In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active device: {device}")

In [ ]:
df = pd.read_csv('fashion-mnist_train.csv')
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2)

In [ ]:
from torchvision.transforms import transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
from PIL import Image
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return self.len(self.features)

    def __getitem__(self, index):
        image = self.features[index].reshape(28,28)
        image = image.astype(np.uint8)
        image = np.stack([image]*3, axis=-1)
        image = Image.fromarray(image)
        image = self.transform(image)

        return image, torch.tensor(self.labels[index], dtype=torch.long)


In [ ]:
train_dataset = CustomDataset(X_train, y_train, transform=custom_transform)
test_dataset = CustomDataset(X_test, y_test, transform=custom_transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size= 32, shuffle = True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle = False, pin_memory=True)

In [ ]:
import torchvision.models as models

vgg16 = models.vgg16(pretrained=True)

In [ ]:
for param in vgg16.features.parameters():
    param.requires_grad = False


In [ ]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088,1024),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(512,10)
)

In [ ]:
learning_rate = 0.001
epochs = 10

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vgg16.classifier.parameters(), lr= learning_rate)

In [ ]:
for epoch in range(epochs):
  total_epoch_loss = 0

  for batch_features, batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = vgg16(batch_features)

        loss = criterion(outputs, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()
        total_epoch_loss += loss.item()

  average_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch: {epoch+1}, Loss: {average_loss}')




In [ ]:
vgg16.eval()

In [ ]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
      batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

      output = vgg16(batch_features)
      _, predicted = torch.max(output, 1)
      total += batch_labels.shape[0]

      correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')

In [ ]:
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in train_loader:
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
    output = vgg16(batch_features)
    _, predicted = torch.max(output, 1)
    total += batch_labels.shape[0]

    correct += (predicted == batch_labels).sum().item()

print(f'Accuracy: {correct/total * 100:.2f}%')